In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    translate,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_payment_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PaymentId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "PaymentKey	string,"
        #     "PaymentUniqueNumber	string,"
        #     "ExposureKey	string,"
        #     "PaymentActualDate	date,"
        #     "PaymentRecordDate	timestamp,"
        #     "PaymentAmount	decimal(15,2),"
        #     "PaymentIntIdentification	string,"
        #     "PaymentCostAmount	decimal(15,2),"
        #     "CoverageAmount	double,"
        #     "DebtAmount	double,"
        #     "MinimumPayment	double,"
        #     "CurrencyCode	string,"
        #     "Currency	string,"
        #     "PaymentTypeCode	string,"
        #     "PaymentType	string,"
        #     "ReasonCode	string,"
        #     "TransactionReasonCode	string,"
        #     "TransactionReason	string,"
        #     "ReportingDate	string,"
        #     "PrincipalAmount	decimal(15,2),"
        #     "CommissionAmount	decimal(15,2),"
        #     "InterestAmount	decimal(15,2),"
        #     "ExpensesAmount	decimal(15,2),"
        #     "InterestURL	string,"
        #     "Createdby	string,"
        #     "IsDownPaymentFlag	boolean,"
        #     "IsReversedFlag	string,"
        #     "PaymentISCOL	string,"
        #     "PaymentRecordType	string,"
        #     "PaymentTypeGroup string,"
        #     "PaymentTypeEng string,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Payment"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        
        

        # ================== fs_payment_leaf ==================

        fs_payment_leaf_extra_filters = None

        if fs_payment_leaf_extra_filters is not None:
            fs_payment_leaf_extra_filters.append(date_filter)
            fs_payment_leaf_final_filters = " AND ".join(fs_payment_leaf_extra_filters)
        else:
            fs_payment_leaf_final_filters = date_filter

            fs_payment_leaf_selected_fields = [
                "CR_DATE",
                "PAY_CODE",
                "PD_DESC",
                "PAY_CREATED",
                "FSPM_AMT_PMT",
                "FSPM_AMT_COST",
                "FSPM_AMT_CVR",
                "FSPM_AMT_DBT",
                "FSPM_AMT_MIN",
                "PAY_CURRENCY",
                "PAY_GENERIC_TEXT_01",
                "PAYR_ID",
                "CASE_ID",
                "SPET_ID",
                "businessdate",
            ]

            fs_payment_leaf = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "qca_fs_payment_leaf",
                fs_payment_leaf_final_filters,
                fs_payment_leaf_selected_fields,
            )
        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

            dim_cases_static_selected_fields = [
                "CASE_ACCOUNT_FOR_SEARCH",
                "CASE_CODE",
                "CASE_CODE_PRIMARY",
   
            ]

            dim_cases_static = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "qca_dim_cases_static",
                dim_cases_static_final_filters,
                dim_cases_static_selected_fields,
            )
        # ================== payment_qca_df ==================

        payment_qca_primary_key_fields = [
            fs_payment_leaf.PAY_CODE,
            lit("qca"),
        ]

        # Removed, we will bring all payments in primary area. Anyone who needs to report payments should join with Exposure
        # FS_CASES_SNAP_dim_cases_static_join_condition = (
        #    FS_CASES_SNAP["CASE_CODE"] == dim_cases_static["CASE_CODE"]
        # ) & (dim_cases_static["CASE_CODE"] == dim_cases_static["CASE_CODE_PRIMARY"])

        fs_payment_leaf_dim_cases_static_join_condition = (
            fs_payment_leaf["CASE_ID"] == dim_cases_static["CASE_CODE"]
        )

        fs_payment_leaf_portfolio_join_condition = (
            fs_payment_leaf["SPET_ID"] == portfolio["portfolio_id_qca"]
        )

        payment_qca_df = (
            fs_payment_leaf.join(
                dim_cases_static,
                fs_payment_leaf_dim_cases_static_join_condition,
                "inner",
            )
            .join(
                portfolio,
                fs_payment_leaf_portfolio_join_condition,
                "left",
            )
        )

        payment_qca_df = payment_qca_df.withColumn(
            "PaymentId",
            sha2(concat_ws(";", *payment_qca_primary_key_fields), 256),
        ).withColumn(
            "PortfolioName",
            when(fs_payment_leaf.SPET_ID.isNull(), "Core Retail")
            .when(fs_payment_leaf.SPET_ID.isin(3, 4), "Core Retail")
            .otherwise(portfolio.PortfolioName)  
        )

        # ================== Selection of Final Columns ==================

        payment_qca_df = payment_qca_df.select(
            payment_qca_df.PaymentId,
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            payment_qca_df.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            payment_qca_df.PAY_CODE.cast("string").alias("PaymentKey"),
            payment_qca_df.PAY_CODE.cast("string").alias("PaymentUniqueNumber"),
            payment_qca_df.CASE_ACCOUNT_FOR_SEARCH.alias("ExposureKey"),
            payment_qca_df.PD_DESC.cast("date").alias("PaymentActualDate"),
            payment_qca_df.PAY_CREATED.alias("PaymentRecordDate"),
            payment_qca_df.FSPM_AMT_PMT.cast("decimal(15,2)").alias("PaymentAmount"),
            lit(None).alias("PaymentIntIdentification"),
            payment_qca_df.FSPM_AMT_COST.cast("Decimal(15,2)").alias(
                "PaymentCostAmount"
            ),
            payment_qca_df.FSPM_AMT_CVR.cast("double").alias("CoverageAmount"),
            payment_qca_df.FSPM_AMT_DBT.cast("double").alias("DebtAmount"),
            payment_qca_df.FSPM_AMT_MIN.cast("double").alias("MinimumPayment"),
            translate(payment_qca_df.PAY_CURRENCY, "Ε", "E").alias("CurrencyCode"),
            translate(payment_qca_df.PAY_CURRENCY, "Ε", "E").alias("Currency"),
            payment_qca_df.PAY_GENERIC_TEXT_01.alias("PaymentTypeCode"),
            payment_qca_df.PAY_GENERIC_TEXT_01.alias("PaymentType"),
            payment_qca_df.PAYR_ID.alias("ReasonCode"),
            lit(None).alias("TransactionReasonCode"),
            lit(None).alias("TransactionReason"),
            lit(None).alias("ReportingDate"),
            lit(None).alias("PrincipalAmount"),
            lit(None).alias("CommissionAmount"),
            lit(None).alias("InterestAmount"),
            lit(None).alias("ExpensesAmount"),
            lit(None).alias("InterestURL"),
            lit(None).alias("Createdby"),
            lit(None).alias("IsDownPaymentFlag"),
            lit(None).alias("IsReversedFlag"),
            lit(None).alias("PaymentISCOL"),
            lit(None).alias("PaymentRecordType"),
            lit(None).alias("PaymentTypeGroup"),
            lit(None).alias("PaymentTypeEng"),
            fs_payment_leaf.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        payment_qca_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_payment_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)